In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [0]:
df = spark.read.table('f1_racing.bronze.brz_results')

display(df.limit(4))

In [0]:
df.count()

In [0]:
df.filter(F.col("position").isNull()).display()

In [0]:
df = df.fillna(-1,
          subset=['constructorId','number','grid','position','positionOrder','laps','milliseconds','fastestLapSpeed'])
df = df.fillna('-1',subset=['position','rank','fastestLap'])


In [0]:
display(df.limit(10))

In [0]:
df.count()

In [0]:
display(df.limit(4))

In [0]:
df.select('time','fastestLapTime').show()

In [0]:

df = df.withColumn(
    "time",
    F.when(
        (F.col("milliseconds") > 0), 
        F.round(F.col("milliseconds") / 60000.0, 3)
    ).otherwise(None)
)

display(df)


In [0]:
lap_parts = F.split(F.col("fastestLapTime"), ":")

df = df.withColumn(
    "fastestLapTime",
    F.when(F.col("fastestLapTime").isNull(), "-1.0")
    .otherwise(
        F.round(
            F.when(F.size(lap_parts) == 3,
                lap_parts.getItem(0).cast(T.DoubleType()) * 60.0 +
                lap_parts.getItem(1).cast(T.DoubleType()) +
                lap_parts.getItem(2).cast(T.DoubleType()) / 60.0
            ).when(F.size(lap_parts) == 2,
                lap_parts.getItem(0).cast(T.DoubleType()) +
                lap_parts.getItem(1).cast(T.DoubleType()) / 60.0
            ).otherwise(None),
            3
        ).cast(T.StringType())
    )
)


In [0]:
df = df.withColumn(
    'time',
    F.when(F.col('time').isNull(), -1.0).otherwise(F.col('time'))
)

df.show()

In [0]:
@F.udf(returnType=T.StringType())
def mapping_positions(position) :
    if position == '-1' :
        return 'Unknown'
    mapping_dict = {
        # Numbers 1 to 20
        "1": "First", "2": "Second", "3": "Third", "4": "Fourth", "5": "Fifth",
        "6": "Sixth", "7": "Seventh", "8": "Eighth", "9": "Ninth", "10": "Tenth",
        "11": "Eleventh", "12": "Twelfth", "13": "Thirteenth", "14": "Fourteenth", "15": "Fifteenth",
        "16": "Sixteenth", "17": "Seventeenth", "18": "Eighteenth", "19": "Nineteenth", "20": "Twentieth",
        "21" : "Twentyfirst",
        
        # Status Letters
        "R": "Retired",
        "D": "Disqualified",
        "N": "Not Classified",
        "W": "Withdrawn",
        "F": "Failed to Qualify"
    }
    if position in mapping_dict:
        return mapping_dict[position]
    else : return 'Unknown'



In [0]:
df.select('positionText').distinct().show()

In [0]:
df = df.withColumn('positionText', mapping_positions(F.col("positionText")))\
    .withColumn('fastestLap',F.col('fastestLap').cast(T.IntegerType()))\
    .withColumn('fastestLapTime',F.col('fastestLapTime').cast(T.DecimalType(10,3)))\
    .withColumn('fastestLapSpeed',F.col('fastestLapSpeed').cast(T.DecimalType(10,3)))\
    .withColumn('position',F.col('position').cast(T.IntegerType()))\
    .drop('source_file','ingested_at')
        

In [0]:
df.display()

In [0]:
df.columns

In [0]:
new_names = {'resultId' : 'result_id',
 'raceId' : 'race_id',
 'driverId' : 'driver_id',
 'constructorId' : 'constructor_id',
 'positionText' : 'position_text',
 'positionOrder' : 'position_order',
 'fastestLap' : 'fastest_lap',
 'fastestLapTime' : 'fastest_lap_time',
 'fastestLapSpeed' : 'fastest_lap_speed',
 'statusId' : 'status_id'}

df = df.withColumnsRenamed(new_names)

display(df)

Databricks filtered table. Run in Databricks to view.

In [0]:
df.columns

In [0]:
df.printSchema()

In [0]:
df.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .option('mergeSchema','true')\
    .saveAsTable('f1_racing.silver.slv_results')